# Single-Player Puzzle MCTS + LLM Optimization Loop

This notebook demonstrates the two-component optimization loop on **single-player puzzles**:

1. **Component 1 — MCTS plays puzzles** with pluggable heuristic functions
2. **Component 2 — LLM analyzes traces** and proposes improved heuristics

Games:
- **Sliding Puzzle (8-puzzle)**: arrange tiles 1-8 in order by sliding them
- **Sokoban**: push boxes onto target positions

The key insight: MCTS with *random* rollouts rarely solves puzzles.
An LLM can discover domain-specific heuristics (e.g., Manhattan distance)
that dramatically improve solve rates.

## Part 1: Understanding the Puzzle Games

In [1]:
# Cell 1: Imports
import sys, os, random, time, re, inspect, textwrap

from mcts import MCTSEngine, TraceLogger, Game, GameState
from mcts.games.sliding_puzzle import SlidingPuzzle, SlidingPuzzleState
from mcts.games.sokoban import Sokoban, SokobanState
from mcts import heuristics as default_heuristics

print('Imports OK')
print(f'Available games: SlidingPuzzle, Sokoban')

Imports OK
Available games: SlidingPuzzle, Sokoban


In [2]:
# Cell 2: Explore the Sliding Puzzle
game = SlidingPuzzle(size=3, scramble_moves=10, max_steps=50)
state = game.new_initial_state()

print(f'Game: {game.name()}')
print(f'Num players: {game.num_players()}')
print(f'State type: {type(state).__name__}')
print()
print('Initial (scrambled) puzzle:')
print(state)
print()
print(f'Legal actions: {state.legal_actions()}')
print(f'Action names: UP=0, DOWN=1, LEFT=2, RIGHT=3  (direction blank moves)')
print(f'Misplaced tiles: {state.misplaced_tiles()}')
print(f'Manhattan distance: {state.manhattan_distance()}')
print(f'Is terminal: {state.is_terminal()}')
print(f'Player: {state.current_player()} (always 0 for single-player)')

Game: Sliding Puzzle 3x3
Num players: 1
State type: SlidingPuzzleState

Initial (scrambled) puzzle:
Step 0/50 | Misplaced: 7 | Manhattan: 8
+--+--+--+
|  |1 |2 |
+--+--+--+
|7 |4 |3 |
+--+--+--+
|5 |8 |6 |
+--+--+--+

Legal actions: [1, 3]
Action names: UP=0, DOWN=1, LEFT=2, RIGHT=3  (direction blank moves)
Misplaced tiles: 7
Manhattan distance: 8
Is terminal: False
Player: 0 (always 0 for single-player)


In [3]:
# Cell 3: Explore Sokoban
from mcts.games.sokoban import LEVELS

print('Available Sokoban levels:')
for name, lines in LEVELS.items():
    game_s = Sokoban(level_name=name)
    s = game_s.new_initial_state()
    print(f'  {name}: {s.num_targets} boxes, '
          f'{len(s.legal_actions())} initial actions')
print()

# Show micro3 (2 boxes, moderate difficulty)
game_s = Sokoban(level_name='micro3', max_steps=100)
state_s = game_s.new_initial_state()
print(f'\nGame: {game_s.name()}')
print(state_s)

Available Sokoban levels:
  micro1: 1 boxes, 4 initial actions
  micro2: 1 boxes, 3 initial actions
  micro3: 2 boxes, 3 initial actions
  micro4: 2 boxes, 4 initial actions
  micro5: 3 boxes, 3 initial actions


Game: Sokoban (micro3)
Step 0/100 | Boxes on target: 0/2 | Total distance: 2
  ####  
###  ###
# $ .  #
# .$ @ #
###  ###
  ####  


## Part 2: Component 1 — MCTS Baseline (No Heuristics)

Run MCTS with **random rollouts** (default heuristics) on the Sliding Puzzle.
With 10 scramble moves, baseline solve rate is ~20-50%.

In [4]:
# Cell 4: Current heuristics (the LLM optimization targets)
engine = MCTSEngine(
    SlidingPuzzle(size=3, scramble_moves=10, max_steps=50),
    iterations=30,
    max_rollout_depth=50,
)

print('=== Current heuristic functions (LLM targets) ===')
for name, src in engine.get_heuristic_source().items():
    print(f'\n--- {name} ---')
    print(src[:300])

=== Current heuristic functions (LLM targets) ===

--- rollout_policy ---
def random_rollout_policy(state: GameState) -> Any:
    """
    Choose an action during the simulation (rollout) phase.

    Args:
        state: Current game state (non-terminal, has legal actions).

    Returns:
        One of state.legal_actions().

    Default: uniform random.
    The LLM can re

--- evaluation ---
def null_evaluation(state: GameState, perspective_player: int) -> float | None:
    """
    Optionally evaluate a non-terminal state and return a value estimate.

    Args:
        state:              Current (possibly non-terminal) game state.
        perspective_player: The player whose perspectiv

--- exploration_weight ---
def default_exploration_weight(root_visits: int) -> float:
    """
    Return the exploration constant C for UCB1.

    Args:
        root_visits: How many total simulations the root has run so far.

    Returns:
        A positive float (typically ~1.4).

    The LLM can imp

In [5]:
# Cell 5: Play baseline games
NUM_GAMES = 30
SCRAMBLE = 10
ITERS = 30

game = SlidingPuzzle(size=3, scramble_moves=SCRAMBLE, max_steps=50)
engine = MCTSEngine(game, iterations=ITERS, max_rollout_depth=50)

print(f'Playing {NUM_GAMES} Sliding Puzzle games '
      f'(scramble={SCRAMBLE}, iters={ITERS}, random rollouts)...')
baseline_stats = engine.play_many(
    num_games=NUM_GAMES,
    clear_table_each_game=True,
)
print(f'\nBaseline stats: {baseline_stats}')
print(f'Solve rate: {baseline_stats["win_rate"]*100:.1f}%')

Playing 30 Sliding Puzzle games (scramble=10, iters=30, random rollouts)...
  Game 3/30: unsolved, moves=50
  Game 6/30: solved, moves=28
  Game 9/30: solved, moves=32
  Game 12/30: unsolved, moves=50
  Game 15/30: unsolved, moves=50
  Game 18/30: unsolved, moves=50
  Game 21/30: unsolved, moves=50
  Game 24/30: unsolved, moves=50
  Game 27/30: solved, moves=36
  Game 30/30: solved, moves=46

Baseline stats: {'total_games': 30, 'wins': 12, 'losses': 0, 'draws': 18, 'win_rate': 0.4}
Solve rate: 40.0%


In [6]:
# Cell 6: Inspect traces — look at unsolved games
report = engine.logger.format_for_llm(max_games=3)
print(report[:2000])

MCTS GAMEPLAY TRACE  ({'total_games': 30, 'wins': 12, 'losses': 0, 'draws': 18, 'win_rate': 0.4})

--- Game 2 | Winner: Player None | MCTS was Player 0 | Moves: 50 | Time: 0.027s ---
  Turn 1: P0 chose action=1 (visits=32, top=[{'action': 1, 'visits': 17, 'avg_value': 0.0}, {'action': 3, 'visits': 17, 'avg_value': 0.0}])
    Board:
      Step 0/50 | Misplaced: 7 | Manhattan: 8
      +--+--+--+
      |  |2 |6 |
      +--+--+--+
      |1 |3 |5 |
      +--+--+--+
      |4 |7 |8 |
      +--+--+--+
  Turn 2: P0 chose action=0 (visits=47, top=[{'action': 0, 'visits': 32, 'avg_value': 0.0}, {'action': 1, 'visits': 24, 'avg_value': 0.0}, {'action': 3, 'visits': 23, 'avg_value': 0.0}])
    Board:
      Step 1/50 | Misplaced: 6 | Manhattan: 7
      +--+--+--+
      |1 |2 |6 |
      +--+--+--+
      |  |3 |5 |
      +--+--+--+
      |4 |7 |8 |
      +--+--+--+
  Turn 3: P0 chose action=1 (visits=62, top=[{'action': 1, 'visits': 47, 'avg_value': 0.0}, {'action': 3, 'visits': 47, 'avg_value': 0.0}]

## Part 3: Component 2 — LLM Analyzes Traces & Proposes Better Heuristics

We send the gameplay traces + current heuristic source code to the LLM
(Qwen 3.5 via Ollama) and ask it to write improved heuristic functions.

In [7]:
# Cell 7: Build the LLM prompt
trace_report = engine.logger.format_for_llm(max_games=5)
heuristic_sources = engine.get_heuristic_source()

prompt = f"""You are an expert game AI engineer optimizing MCTS heuristics for a SINGLE-PLAYER puzzle.

## Game: Sliding Puzzle (8-puzzle)
- 3x3 grid with tiles 1-8 and one blank space
- Actions: slide the blank UP(0), DOWN(1), LEFT(2), RIGHT(3)
- Goal: arrange tiles as [1,2,3,4,5,6,7,8,_] (blank at bottom-right)
- The puzzle state has helper methods:
  - state.misplaced_tiles() -> int (tiles not in goal position)
  - state.manhattan_distance() -> int (sum of Manhattan distances to goals)
  - state.board -> list[int] (flat board, 0=blank)
  - state.size -> int (3 for 3x3)
  - state.goal -> list[int] (the goal configuration)
  - state.blank_pos -> int (index of blank)

## Current Heuristics (MCTS is solving only ~{baseline_stats['win_rate']*100:.0f}% of puzzles):

### evaluation (most impactful for puzzles — short-circuits random rollouts):
{heuristic_sources['evaluation']}

### rollout_policy:
{heuristic_sources['rollout_policy']}

## Gameplay Traces (unsolved puzzles):
{trace_report[:3000]}

## Task
The `evaluation` function is the most impactful heuristic for puzzles.
Currently it returns None (pure random rollout), which almost never finds solutions.

Write an improved `evaluation` function that estimates how close a state is
to being solved, so MCTS can guide its search effectively.

Return a float in [0, 1] where 1.0 = solved and 0.0 = far from solution.
Return None only if you want the rollout to continue (not recommended).

IMPORTANT: Write ONLY the function definition. The function signature must be:
  def evaluation(state: GameState, perspective_player: int) -> float | None:

Put your code in a ```python code block.
"""

print(f'Prompt length: {len(prompt)} chars')
print('\nFirst 500 chars of prompt:')
print(prompt[:500])

Prompt length: 5594 chars

First 500 chars of prompt:
You are an expert game AI engineer optimizing MCTS heuristics for a SINGLE-PLAYER puzzle.

## Game: Sliding Puzzle (8-puzzle)
- 3x3 grid with tiles 1-8 and one blank space
- Actions: slide the blank UP(0), DOWN(1), LEFT(2), RIGHT(3)
- Goal: arrange tiles as [1,2,3,4,5,6,7,8,_] (blank at bottom-right)
- The puzzle state has helper methods:
  - state.misplaced_tiles() -> int (tiles not in goal position)
  - state.manhattan_distance() -> int (sum of Manhattan distances to goals)
  - state.board -> 


In [8]:
# Cell 8: Call Qwen 3.5 via Ollama
import requests, json

OLLAMA_URL = 'http://localhost:11434/api/chat'
MODEL = 'qwen3.5:35b'

resp = requests.post(OLLAMA_URL, json={
    'model': MODEL,
    'messages': [{'role': 'user', 'content': prompt}],
    'stream': False,
}, timeout=300)
resp.raise_for_status()
data = resp.json()

# Qwen 3.5 may put code in 'thinking' field with empty 'content'
content = data['message'].get('content', '')
thinking = data['message'].get('thinking', '')
full_response = content + '\n' + thinking

print('=== LLM Response (first 1500 chars) ===')
print(content[:1500] if content.strip() else thinking[:1500])

=== LLM Response (first 1500 chars) ===
```python
def evaluation(state: GameState, perspective_player: int) -> float | None:
    """
    Evaluates the current state using Manhattan Distance heuristic.
    Returns 1.0 for solved states, and a value approaching 0.0 for states far from the goal.
    """
    md = state.manhattan_distance()
    if md == 0:
        return 1.0
    
    # Normalize Manhattan Distance to [0, 1] range.
    # 1.0 / (1 + md) maps MD=0 to 1.0 and larger MD to values close to 0.0.
    return 1.0 / (1.0 + md)
```


In [9]:
# Cell 9: Extract and load the improved evaluation function
import re

def extract_python_code(text: str) -> str:
    """Extract python code from markdown code blocks."""
    blocks = re.findall(r'```python\s*\n(.*?)```', text, re.DOTALL)
    if blocks:
        return blocks[0].strip()
    blocks = re.findall(r'```\s*\n(.*?)```', text, re.DOTALL)
    if blocks:
        return blocks[0].strip()
    return text.strip()

code = extract_python_code(full_response)
print('=== Extracted Code ===')
print(code)

# Execute the code to define the function
namespace = {
    'GameState': GameState,
    'SlidingPuzzleState': SlidingPuzzleState,
    'math': __import__('math'),
    'random': __import__('random'),
}
exec(code, namespace)

# Find the evaluation function
improved_eval = namespace.get('evaluation')
if improved_eval is None:
    # Maybe the function has a different name
    for k, v in namespace.items():
        if callable(v) and k != '__builtins__':
            improved_eval = v
            break

assert improved_eval is not None, 'Failed to extract evaluation function!'
print(f'\nLoaded function: {improved_eval.__name__}')

# Quick sanity test
test_state = SlidingPuzzle(size=3, scramble_moves=5).new_initial_state()
val = improved_eval(test_state, 0)
print(f'Test evaluation on scrambled state: {val}')

=== Extracted Code ===
def evaluation(state: GameState, perspective_player: int) -> float | None:
    """
    Evaluates the current state using Manhattan Distance heuristic.
    Returns 1.0 for solved states, and a value approaching 0.0 for states far from the goal.
    """
    md = state.manhattan_distance()
    if md == 0:
        return 1.0
    
    # Normalize Manhattan Distance to [0, 1] range.
    # 1.0 / (1 + md) maps MD=0 to 1.0 and larger MD to values close to 0.0.
    return 1.0 / (1.0 + md)

Loaded function: evaluation
Test evaluation on scrambled state: 0.16666666666666666


## Part 4: Hot-Swap & Re-Evaluate

Replace the default evaluation with the LLM's version and re-run the same puzzles.

In [10]:
# Cell 10: Compare baseline vs LLM-improved
game2 = SlidingPuzzle(size=3, scramble_moves=SCRAMBLE, max_steps=50)
engine2 = MCTSEngine(game2, iterations=ITERS, max_rollout_depth=50)

# Hot-swap the evaluation heuristic
engine2.set_heuristic('evaluation', improved_eval)
print(f'Swapped evaluation -> {improved_eval.__name__}')

print(f'\nPlaying {NUM_GAMES} games with LLM-improved evaluation...')
improved_stats = engine2.play_many(
    num_games=NUM_GAMES,
    clear_table_each_game=True,
)

print(f'\n{"="*50}')
print(f'BASELINE  solve rate: {baseline_stats["win_rate"]*100:.1f}% '
      f'({baseline_stats["wins"]}/{baseline_stats["total_games"]})')
print(f'IMPROVED  solve rate: {improved_stats["win_rate"]*100:.1f}% '
      f'({improved_stats["wins"]}/{improved_stats["total_games"]})')
delta = (improved_stats['win_rate'] - baseline_stats['win_rate']) * 100
print(f'IMPROVEMENT: {delta:+.1f} percentage points')
print(f'{"="*50}')

Swapped evaluation -> evaluation

Playing 30 games with LLM-improved evaluation...
  Game 3/30: solved, moves=36
  Game 6/30: solved, moves=40
  Game 9/30: unsolved, moves=50
  Game 12/30: solved, moves=4
  Game 15/30: unsolved, moves=50
  Game 18/30: solved, moves=28
  Game 21/30: solved, moves=6
  Game 24/30: unsolved, moves=50
  Game 27/30: unsolved, moves=50
  Game 30/30: unsolved, moves=50

BASELINE  solve rate: 40.0% (12/30)
IMPROVED  solve rate: 50.0% (15/30)
IMPROVEMENT: +10.0 percentage points


## Part 5: Full Automated Multi-Round Optimization Loop

Run multiple rounds of:
1. Play puzzles with current heuristics
2. Send traces to LLM -> get improved code
3. Hot-swap and measure improvement

In [13]:
# Cell 11: Automated loop (with "keep best" safeguard)
import requests, json, re, time

OLLAMA_URL = 'http://localhost:11434/api/chat'
MODEL = 'qwen3.5:35b'
NUM_ROUNDS = 3
GAMES_PER_ROUND = 30
SCRAMBLE_MOVES = 10
MCTS_ITERS = 30
ROLLOUT_DEPTH = 50
VALIDATION_GAMES = 30   # games to validate a new heuristic before adopting

def extract_python_code(text: str) -> str:
    blocks = re.findall(r'```python\s*\n(.*?)```', text, re.DOTALL)
    if blocks: return blocks[0].strip()
    blocks = re.findall(r'```\s*\n(.*?)```', text, re.DOTALL)
    if blocks: return blocks[0].strip()
    return text.strip()

def call_llm(prompt_text: str) -> str:
    resp = requests.post(OLLAMA_URL, json={
        'model': MODEL,
        'messages': [{'role': 'user', 'content': prompt_text}],
        'stream': False,
    }, timeout=600)   # 10 min timeout for 35B model
    resp.raise_for_status()
    data = resp.json()
    content = data['message'].get('content', '')
    thinking = data['message'].get('thinking', '')
    return content + '\n' + thinking

def build_puzzle_prompt(engine, stats, prev_code=None):
    trace = engine.logger.format_for_llm(max_games=5)
    sources = engine.get_heuristic_source()
    
    prev_section = ''
    if prev_code:
        prev_section = f"""\n## Previous Attempt (solve rate {stats['win_rate']*100:.0f}%):\n```python\n{prev_code}\n```\nThis needs improvement. Try a different or better approach.\n"""
    
    return f"""You are an expert game AI engineer optimizing MCTS heuristics for a SINGLE-PLAYER puzzle.

## Game: Sliding Puzzle (8-puzzle)
- 3x3 grid with tiles 1-8 and one blank space
- Actions: slide the blank UP(0), DOWN(1), LEFT(2), RIGHT(3)
- Goal: arrange tiles as [1,2,3,4,5,6,7,8,_] (blank at bottom-right)
- The puzzle state has helper methods:
  - state.misplaced_tiles() -> int (tiles not in goal position)
  - state.manhattan_distance() -> int (sum of Manhattan distances to goals)
  - state.board -> list[int] (flat board, 0=blank)
  - state.size -> int (3 for 3x3)
  - state.goal -> list[int] (the goal configuration)
  - state.blank_pos -> int (index of blank)
  - state.legal_actions() -> list[int]

## Current Performance: {stats['win_rate']*100:.0f}% solve rate

### Current evaluation function:
{sources['evaluation']}
{prev_section}
## Gameplay Traces (unsolved puzzles -- MCTS couldn't solve these):
{trace[:3000]}

## Task
Write an improved `evaluation` function. This function is called during MCTS
simulation -- it short-circuits the random rollout by returning a value estimate.

CRITICAL DESIGN RULES:
1. Return a float in [0, 1] where 1.0 = solved and 0.0 = far from solution.
2. Values must have STRONG contrast: near-solved states ~0.9+, scrambled states ~0.05-0.1.
   If values are compressed (e.g. all in 0.5-0.9), MCTS cannot differentiate states.
3. Manhattan distance divided by (1 + md) works well. Avoid (max_md - md)/max_md shape.
4. Keep it simple -- complex functions often perform worse than simple ones.

Function signature:
  def evaluation(state: GameState, perspective_player: int) -> float | None:

Put your code in a ```python code block. Write ONLY the function.
"""

def validate_heuristic(eval_fn, label="candidate"):
    """Play VALIDATION_GAMES and return solve rate."""
    g = SlidingPuzzle(size=3, scramble_moves=SCRAMBLE_MOVES, max_steps=50)
    e = MCTSEngine(g, iterations=MCTS_ITERS, max_rollout_depth=ROLLOUT_DEPTH)
    if eval_fn is not None:
        e.set_heuristic('evaluation', eval_fn)
    stats = e.play_many(num_games=VALIDATION_GAMES, clear_table_each_game=True, verbose=False)
    return stats['win_rate']

# --- Establish baseline ---
print('Measuring baseline (random rollouts, no evaluation)...')
best_eval = None
best_rate = validate_heuristic(None, "baseline")
print(f'Baseline solve rate: {best_rate*100:.1f}%')

# --- Main loop ---
results = [{'round': 0, 'rate': best_rate, 'label': 'baseline'}]
prev_code = None

for round_num in range(1, NUM_ROUNDS + 1):
    print(f'\n{"="*60}')
    print(f'ROUND {round_num}/{NUM_ROUNDS}')
    print(f'{"="*60}')
    
    # 1. Play games with BEST heuristic (for traces + stats)
    game = SlidingPuzzle(size=3, scramble_moves=SCRAMBLE_MOVES, max_steps=50)
    engine = MCTSEngine(game, iterations=MCTS_ITERS, max_rollout_depth=ROLLOUT_DEPTH)
    if best_eval:
        engine.set_heuristic('evaluation', best_eval)
    
    t0 = time.time()
    stats = engine.play_many(num_games=GAMES_PER_ROUND, clear_table_each_game=True)
    play_time = time.time() - t0
    current_rate = stats['win_rate']
    print(f'  Play: {current_rate*100:.1f}% solved ({play_time:.1f}s)')
    
    # 2. Generate prompt and call LLM
    prompt = build_puzzle_prompt(engine, stats, prev_code)
    print(f'  Calling LLM ({len(prompt)} char prompt)...')
    t1 = time.time()
    response = call_llm(prompt)
    llm_time = time.time() - t1
    print(f'  LLM responded in {llm_time:.1f}s')
    
    # 3. Extract and load new evaluation
    code = extract_python_code(response)
    prev_code = code
    print(f'  Extracted code ({len(code)} chars)')
    
    ns = {
        'GameState': GameState,
        'SlidingPuzzleState': SlidingPuzzleState,
        'math': __import__('math'),
        'random': __import__('random'),
    }
    try:
        exec(code, ns)
        new_eval = ns.get('evaluation')
        if new_eval is None:
            for k, v in ns.items():
                if callable(v) and k not in ('__builtins__', 'GameState', 'SlidingPuzzleState'):
                    new_eval = v
                    break
        # Sanity test
        test_s = SlidingPuzzle(size=3, scramble_moves=5).new_initial_state()
        val = new_eval(test_s, 0)
        assert isinstance(val, (int, float, type(None)))
        
        # VALIDATE: only adopt if better than current best
        cand_rate = validate_heuristic(new_eval, "candidate")
        print(f'  Candidate solve rate: {cand_rate*100:.1f}% (best so far: {best_rate*100:.1f}%)')
        
        if cand_rate > best_rate:
            best_eval = new_eval
            best_rate = cand_rate
            results.append({'round': round_num, 'rate': cand_rate, 'label': 'adopted'})
            print(f'  -> ADOPTED new evaluation (improvement!)')
        else:
            results.append({'round': round_num, 'rate': best_rate, 'label': 'rejected'})
            print(f'  -> Rejected -- not better than current best')
    except Exception as e:
        results.append({'round': round_num, 'rate': best_rate, 'label': 'error'})
        print(f'  FAIL to load: {e}')

# Final summary
print(f'\n{"="*60}')
print('RESULTS')
print(f'{"="*60}')
print(f'\nSolve rate progression:')
for r in results:
    bar = '#' * int(r['rate'] * 40)
    label = f'  R{r["round"]} ({r["label"]})'
    print(f'{label:>20s}: {r["rate"]*100:5.1f}% {bar}')
print(f'\nTotal improvement: {(best_rate - results[0]["rate"])*100:+.1f} pp')

Measuring baseline (random rollouts, no evaluation)...
  Game 3/30: solved, moves=40
  Game 6/30: unsolved, moves=50
  Game 9/30: unsolved, moves=50
  Game 12/30: unsolved, moves=50
  Game 15/30: solved, moves=36
  Game 18/30: unsolved, moves=50
  Game 21/30: unsolved, moves=50
  Game 24/30: unsolved, moves=50
  Game 27/30: solved, moves=46
  Game 30/30: unsolved, moves=50
Baseline solve rate: 36.7%

ROUND 1/3
  Game 3/30: unsolved, moves=50
  Game 6/30: unsolved, moves=50
  Game 9/30: unsolved, moves=50
  Game 12/30: solved, moves=50
  Game 15/30: unsolved, moves=50
  Game 18/30: unsolved, moves=50
  Game 21/30: unsolved, moves=50
  Game 24/30: solved, moves=34
  Game 27/30: unsolved, moves=50
  Game 30/30: solved, moves=30
  Play: 36.7% solved (0.6s)
  Calling LLM (5230 char prompt)...
  LLM responded in 311.7s
  Extracted code (545 chars)
  Game 3/30: unsolved, moves=50
  Game 6/30: unsolved, moves=50
  Game 9/30: solved, moves=42
  Game 12/30: unsolved, moves=50
  Game 15/30: unsol

## Part 6 (Optional): Test on Sokoban

Same optimization loop, different game.

In [14]:
# Cell 12: Sokoban baseline
sok_game = Sokoban(level_name='micro3', max_steps=100)
sok_engine = MCTSEngine(sok_game, iterations=30, max_rollout_depth=30)

print('Sokoban micro3 baseline (2 boxes)...')
sok_stats = sok_engine.play_many(num_games=10, clear_table_each_game=True)
print(f'Solve rate: {sok_stats["win_rate"]*100:.0f}%')
print(f'\nUnsolved trace:')
print(sok_engine.logger.format_for_llm(max_games=2)[:1500])

Sokoban micro3 baseline (2 boxes)...
  Game 1/10: unsolved, moves=100
  Game 2/10: unsolved, moves=100
  Game 3/10: unsolved, moves=100
  Game 4/10: unsolved, moves=100
  Game 5/10: unsolved, moves=100
  Game 6/10: unsolved, moves=100
  Game 7/10: unsolved, moves=100
  Game 8/10: unsolved, moves=100
  Game 9/10: unsolved, moves=100
  Game 10/10: unsolved, moves=100
Solve rate: 0%

Unsolved trace:
MCTS GAMEPLAY TRACE  ({'total_games': 10, 'wins': 0, 'losses': 0, 'draws': 10, 'win_rate': 0.0})

--- Game 1 | Winner: Player None | MCTS was Player 0 | Moves: 100 | Time: 0.068s ---
  Turn 1: P0 chose action=0 (visits=33, top=[{'action': 0, 'visits': 12, 'avg_value': 0.0}, {'action': 3, 'visits': 12, 'avg_value': 0.0}, {'action': 2, 'visits': 11, 'avg_value': 0.0}])
    Board:
      Step 0/100 | Boxes on target: 0/2 | Total distance: 2
        ####  
      ###  ###
      # $ .  #
      # .$ @ #
      ###  ###
        ####  
  Turn 2: P0 chose action=1 (visits=42, top=[{'action': 1, 'visits': 